# Profile and validate registered source tables

Compare missing and empty fields across inputs. The full metric tables retain field-level values, denominators, distributions and duplicate counts for inspection.

Select the dataset through the Compose environment or set `MEDLENS_DATASET` before opening the kernel. The resolved configuration determines the enabled tables and policies.

In [ ]:
%matplotlib inline
from medlens.notebook import workspace, notebook_run
import matplotlib.pyplot as plt

research = workspace()

The run records its input versions, resolved configuration, and producing stages. Set `MEDLENS_PARENT` to replay from a specific published manifest. The default walkthrough uses the preceding notebook checkpoint for this configuration.

In [ ]:
run = notebook_run(research, 'profiling')
print("Run", run["run_id"], "Parent", run["parent"])

In [ ]:
from pyspark.sql import functions as F
missing = {}
for stage in research.config.stages.values():
    if stage.enabled and stage.op == "profile":
        profile = research.frame(run, stage.output)
        total = profile.where(F.col("metric").isin("nulls", "empty")).agg(F.sum(F.col("value").cast("long"))).first()[0]
        missing[stage.subject] = total or 0
plt.barh(list(missing), list(missing.values()))
plt.xlabel("Null or empty cells")
plt.tight_layout()
plt.show()
print("Profile and quality tables", [s["stage"] for s in run["stages"]])

In [ ]:
research.close()